# Algo Wheel Experiments
Run a small parameter wheel and rank candidates by net Sharpe and promotion status.


In [ ]:
import sys
from pathlib import Path

repo = Path.cwd()
if str(repo / "python") not in sys.path:
    sys.path.insert(0, str(repo / "python"))

import openquant


## Setup
Deterministic synthetic dataset for reproducible ranking.


In [ ]:
ds = openquant.research.make_synthetic_futures_dataset(n_bars=192, seed=7)

configs = [
    {"step_size": 0.05, "cusum_threshold": 0.0010},
    {"step_size": 0.10, "cusum_threshold": 0.0012},
    {"step_size": 0.15, "cusum_threshold": 0.0012, "commission_bps": 2.5},
    {"step_size": 0.08, "cusum_threshold": 0.0015, "min_net_sharpe": 0.4},
]
run_names = ["base", "coarser_step", "higher_costs", "stricter_gate"]


## Execute Grid


In [ ]:
grid = openquant.research.run_flywheel_grid(ds, configs=configs, run_names=run_names)
leaderboard = grid["leaderboard"]
leaderboard


## Top Run Replay


In [ ]:
top = leaderboard.row(0, named=True)
top


In [ ]:
top_run = [r for r in grid["runs"] if r["run_name"] == top["run_name"]][0]
{
    "run_name": top_run["run_name"],
    "summary": top_run["output"]["summary"].to_dicts(),
    "promotion": top_run["output"]["promotion"],
}


## Analysis
- Use this notebook for discovery ranking only.
- Promote candidates into `experiments/run_pipeline.py --grid-config ...` for artifact-grade runs.
